In [3]:
# ==========================================================
# ALGORITHME CART FROM SCRATCH EN PYTHON
# ==========================================================
#
# CART = Classification And Regression Tree
#
# ----------------------------------------------------------
# CART EST UTILISÉ POUR :
# ----------------------------------------------------------
#
# ✔ Classification
# ✔ Régression
#
# Exemple classification :
#
# -> Spam / Non Spam
# -> Oui / Non
# -> Malade / Sain
#
# Exemple régression :
#
# -> prédire un prix
# -> prédire une température
#
# ----------------------------------------------------------
# DIFFÉRENCE ENTRE :
# ----------------------------------------------------------
#
# ID3 / C4.5 :
# -> utilisent ENTROPIE
#
# CART :
# -> utilise GINI IMPURITY
#
# ----------------------------------------------------------
# AUTRE DIFFÉRENCE IMPORTANTE
# ----------------------------------------------------------
#
# ID3 et C4.5 :
# -> plusieurs branches possibles
#
# CART :
# -> arbre binaire uniquement
#
# Exemple :
#
# Gauche  -> condition vraie
# Droite  -> condition fausse
#
# ----------------------------------------------------------
# DANS CE CODE ON IMPLÉMENTE :
# ----------------------------------------------------------
#
# ✔ Indice de Gini
# ✔ Split du dataset
# ✔ Choix du meilleur split
# ✔ Construction récursive
# ✔ Prédiction
#

#
# ==========================================================

# ==========================================================
# IMPORTATION DES LIBRARIES
# ==========================================================

# pandas :
# utilisé pour manipuler les tableaux
import pandas as pd

# numpy :
# utilisé pour les calculs mathématiques
import numpy as np

# Counter :
# utilisé pour compter les classes
from collections import Counter

# ==========================================================
# 1. DATASET
# ==========================================================

# ----------------------------------------------------------
# OBJECTIF :
# ----------------------------------------------------------
#
# prédire si on joue au tennis
#
# Variable cible :
#
# Jouer = Oui / Non
#
# ----------------------------------------------------------
# FEATURES :
# ----------------------------------------------------------
#
# Meteo
# Temperature
# Humidite
# Vent
#
# ----------------------------------------------------------

data = {

    # ------------------------------------------------------
    # FEATURE : MÉTÉO
    # ------------------------------------------------------
    #
    # Soleil
    # Nuageux
    # Pluie
    #
    # ------------------------------------------------------

    'Meteo': [

        'Soleil',
        'Soleil',
        'Nuageux',
        'Pluie',
        'Pluie',
        'Pluie',
        'Nuageux',
        'Soleil',
        'Soleil',
        'Pluie'
    ],

    # ------------------------------------------------------
    # FEATURE : TEMPÉRATURE
    # ------------------------------------------------------

    'Temperature': [

        'Chaud',
        'Chaud',
        'Chaud',
        'Moyen',
        'Froid',
        'Froid',
        'Froid',
        'Moyen',
        'Froid',
        'Moyen'
    ],

    # ------------------------------------------------------
    # FEATURE : HUMIDITÉ
    # ------------------------------------------------------

    'Humidite': [

        'Haute',
        'Haute',
        'Haute',
        'Haute',
        'Normale',
        'Normale',
        'Normale',
        'Haute',
        'Normale',
        'Normale'
    ],

    # ------------------------------------------------------
    # FEATURE : VENT
    # ------------------------------------------------------

    'Vent': [

        'Faible',
        'Fort',
        'Faible',
        'Faible',
        'Faible',
        'Fort',
        'Fort',
        'Faible',
        'Faible',
        'Faible'
    ],

    # ------------------------------------------------------
    # VARIABLE CIBLE
    # ------------------------------------------------------
    #
    # Oui = jouer
    # Non = ne pas jouer
    #
    # ------------------------------------------------------

    'Jouer': [

        'Non',
        'Non',
        'Oui',
        'Oui',
        'Oui',
        'Non',
        'Oui',
        'Non',
        'Oui',
        'Oui'
    ]
}

# ----------------------------------------------------------
# Transformer dictionnaire -> DataFrame
# ----------------------------------------------------------

df = pd.DataFrame(data)

# ----------------------------------------------------------
# Affichage dataset
# ----------------------------------------------------------

print("===== DATASET =====")

print(df)

# ==========================================================
# 2. GINI IMPURITY
# ==========================================================

def gini(y):

    """
    ------------------------------------------------------
    GINI IMPURITY
    ------------------------------------------------------

    Gini mesure :
    le niveau d'impureté des données.

    ------------------------------------------------------
    CAS 1 :
    ------------------------------------------------------

    Oui Oui Oui Oui

    -> Gini = 0
    -> données parfaitement pures

    ------------------------------------------------------
    CAS 2 :
    ------------------------------------------------------

    Oui Non Oui Non

    -> Gini élevé
    -> données mélangées

    ------------------------------------------------------
    OBJECTIF DE CART :
    ------------------------------------------------------

    minimiser le Gini.

    ------------------------------------------------------
    PLUS GINI EST PETIT :
    ------------------------------------------------------

    -> meilleure séparation

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    Gini = 1 - Σ p(x)^2

    ------------------------------------------------------
    p(x)
    ------------------------------------------------------

    probabilité d'une classe

    Exemple :

    Oui = 6/10
    Non = 4/10
    """

    # ------------------------------------------------------
    # Compter les classes
    # ------------------------------------------------------
    #
    # Exemple :
    #
    # Oui = 6
    # Non = 4
    #
    # ------------------------------------------------------

    counts = Counter(y)

    # ------------------------------------------------------
    # Nombre total d'exemples
    # ------------------------------------------------------

    total = len(y)

    # ------------------------------------------------------
    # Initialiser impurity à 1
    # ------------------------------------------------------

    impurity = 1

    # ------------------------------------------------------
    # Parcourir chaque classe
    # ------------------------------------------------------

    for count in counts.values():

        # --------------------------------------------------
        # Calcul probabilité
        # --------------------------------------------------

        p = count / total

        # --------------------------------------------------
        # Formule du Gini
        # --------------------------------------------------

        impurity -= p ** 2

    # ------------------------------------------------------
    # Retourner Gini final
    # ------------------------------------------------------

    return impurity

# ==========================================================
# 3. SPLIT DU DATASET
# ==========================================================

def split_dataset(data, feature, value):

    """
    ------------------------------------------------------
    Cette fonction divise le dataset
    en deux parties.
    ------------------------------------------------------

    LEFT  :
    feature == value

    RIGHT :
    feature != value

    ------------------------------------------------------
    Exemple :
    ------------------------------------------------------

    feature = Meteo
    value   = Soleil

    LEFT  -> Meteo = Soleil
    RIGHT -> Meteo != Soleil
    """

    # ------------------------------------------------------
    # Partie gauche
    # ------------------------------------------------------

    left = data[data[feature] == value]

    # ------------------------------------------------------
    # Partie droite
    # ------------------------------------------------------

    right = data[data[feature] != value]

    return left, right

# ==========================================================
# 4. CALCUL GINI DU SPLIT
# ==========================================================

def split_gini(data, feature, target):

    """
    ------------------------------------------------------
    Cette fonction cherche
    le meilleur split possible
    pour une feature.
    ------------------------------------------------------

    Elle teste chaque valeur
    de la feature.

    Puis elle calcule :
    -> Gini gauche
    -> Gini droite
    -> Gini total pondéré
    """

    # ------------------------------------------------------
    # Valeurs uniques
    # ------------------------------------------------------

    values = data[feature].unique()

    # ------------------------------------------------------
    # Initialisation meilleur Gini
    # ------------------------------------------------------
    #
    # float('inf')
    #
    # = très grande valeur
    #
    # ------------------------------------------------------

    best_gini = float('inf')

    # ------------------------------------------------------
    # Meilleure valeur du split
    # ------------------------------------------------------

    best_value = None

    # ------------------------------------------------------
    # Tester chaque valeur
    # ------------------------------------------------------

    for value in values:

        # --------------------------------------------------
        # Diviser dataset
        # --------------------------------------------------

        left, right = split_dataset(
            data,
            feature,
            value
        )

        # --------------------------------------------------
        # Éviter division vide
        # ------------------------------------------------------
        #
        # Si une branche est vide,
        # le split est inutile.
        #
        # --------------------------------------------------

        if len(left) == 0 or len(right) == 0:

            continue

        # --------------------------------------------------
        # Gini partie gauche
        # --------------------------------------------------

        gini_left = gini(left[target])

        # --------------------------------------------------
        # Gini partie droite
        # --------------------------------------------------

        gini_right = gini(right[target])

        # --------------------------------------------------
        # Calcul poids gauche
        # --------------------------------------------------

        weight_left = len(left) / len(data)

        # --------------------------------------------------
        # Calcul poids droite
        # --------------------------------------------------

        weight_right = len(right) / len(data)

        # --------------------------------------------------
        # Gini pondéré total
        # ------------------------------------------------------
        #
        # Formule :
        #
        # GiniTotal =
        #
        # poids_gauche × gini_gauche
        # +
        # poids_droite × gini_droite
        #
        # --------------------------------------------------

        total_gini = (

            weight_left * gini_left +

            weight_right * gini_right
        )

        # --------------------------------------------------
        # Garder meilleur split
        # ------------------------------------------------------
        #
        # CART cherche :
        #
        # le PLUS PETIT Gini
        #
        # --------------------------------------------------

        if total_gini < best_gini:

            best_gini = total_gini

            best_value = value

    # ------------------------------------------------------
    # Retourner meilleur résultat
    # ------------------------------------------------------

    return best_gini, best_value

# ==========================================================
# 5. CHOISIR MEILLEURE FEATURE
# ==========================================================

def best_split(data, features, target):

    """
    ------------------------------------------------------
    Cette fonction teste
    toutes les features.
    ------------------------------------------------------

    Puis elle choisit :
    -> la feature avec le plus petit Gini
    """

    # ------------------------------------------------------
    # Meilleure feature
    # ------------------------------------------------------

    best_feature_name = None

    # ------------------------------------------------------
    # Meilleure valeur
    # ------------------------------------------------------

    best_value = None

    # ------------------------------------------------------
    # Meilleur score Gini
    # ------------------------------------------------------

    best_gini_score = float('inf')

    print("\n===== SCORES GINI =====")

    # ------------------------------------------------------
    # Tester chaque feature
    # ------------------------------------------------------

    for feature in features:

        gini_score, value = split_gini(
            data,
            feature,
            target
        )

        print(f"{feature} -> Gini : {gini_score:.4f}")

        # --------------------------------------------------
        # Garder meilleur split
        # --------------------------------------------------

        if gini_score < best_gini_score:

            best_gini_score = gini_score

            best_feature_name = feature

            best_value = value

    # ------------------------------------------------------
    # Retourner meilleur split
    # ------------------------------------------------------

    return best_feature_name, best_value

# ==========================================================
# 6. CONSTRUCTION ARBRE CART
# ==========================================================

def cart(data, features, target):

    """
    ------------------------------------------------------
    Fonction principale CART
    ------------------------------------------------------

    Construction récursive de l'arbre.
    """

    # ------------------------------------------------------
    # Classes présentes
    # ------------------------------------------------------

    labels = data[target]

    # ======================================================
    # CAS D'ARRÊT 1
    # ======================================================
    #
    # Toutes les classes identiques
    #
    # Exemple :
    #
    # Oui Oui Oui
    #
    # -> feuille finale
    #
    # ======================================================

    if len(np.unique(labels)) == 1:

        return labels.iloc[0]

    # ======================================================
    # CAS D'ARRÊT 2
    # ======================================================
    #
    # Plus de features disponibles
    #
    # ======================================================

    if len(features) == 0:

        # retourner classe majoritaire

        return labels.mode()[0]

    # ======================================================
    # Recherche meilleur split
    # ======================================================

    best_feature_name, best_value = best_split(

        data,
        features,
        target
    )

    # ------------------------------------------------------
    # Aucun split trouvé
    # ------------------------------------------------------

    if best_feature_name is None:

        return labels.mode()[0]

    # ======================================================
    # Création arbre
    # ======================================================

    tree = {

        best_feature_name: {}
    }

    # ======================================================
    # SPLIT BINAIRE
    # ======================================================

    left, right = split_dataset(

        data,
        best_feature_name,
        best_value
    )

    # ======================================================
    # Supprimer feature utilisée
    # ======================================================

    remaining_features = [

        f for f in features

        if f != best_feature_name
    ]

    # ======================================================
    # BRANCHE GAUCHE
    # ======================================================
    #
    # condition :
    #
    # feature == value
    #
    # ======================================================

    tree[best_feature_name][f"== {best_value}"] = cart(

        left,
        remaining_features,
        target
    )

    # ======================================================
    # BRANCHE DROITE
    # ======================================================
    #
    # condition :
    #
    # feature != value
    #
    # ======================================================

    tree[best_feature_name][f"!= {best_value}"] = cart(

        right,
        remaining_features,
        target
    )

    # ------------------------------------------------------
    # Retourner arbre final
    # ------------------------------------------------------

    return tree

# ==========================================================
# 7. ENTRAINEMENT
# ==========================================================

# ----------------------------------------------------------
# Liste des features
# ----------------------------------------------------------

features = [

    'Meteo',
    'Temperature',
    'Humidite',
    'Vent'
]

# ----------------------------------------------------------
# Variable cible
# ----------------------------------------------------------

target = 'Jouer'

# ----------------------------------------------------------
# Construction arbre
# ----------------------------------------------------------

tree = cart(
    df,
    features,
    target
)

# ==========================================================
# 8. AFFICHAGE
# ==========================================================

print("\n===== ARBRE CART =====")

print(tree)

# ==========================================================
# 9. PRÉDICTION
# ==========================================================

def predict(tree, sample):

    """
    ------------------------------------------------------
    Fonction de prédiction
    ------------------------------------------------------

    Elle parcourt l'arbre
    jusqu'à atteindre une feuille finale.
    """

    # ------------------------------------------------------
    # Récupérer racine
    # ------------------------------------------------------

    root = list(tree.keys())[0]

    # ------------------------------------------------------
    # Récupérer branches
    # ------------------------------------------------------

    branches = tree[root]

    # ------------------------------------------------------
    # Parcourir chaque branche
    # ------------------------------------------------------

    for condition, subtree in branches.items():

        # --------------------------------------------------
        # Séparer opérateur et valeur
        # --------------------------------------------------
        #
        # Exemple :
        #
        # "== Soleil"
        #
        # operator = ==
        # value    = Soleil
        #
        # --------------------------------------------------

        operator, value = condition.split(" ", 1)

        # --------------------------------------------------
        # Valeur de l'exemple
        # --------------------------------------------------

        sample_value = sample[root]

        # ==================================================
        # CONDITION ==
        # ==================================================

        if operator == "==":

            if sample_value == value:

                # feuille finale
                if not isinstance(subtree, dict):

                    return subtree

                # continuer récursivement
                return predict(subtree, sample)

        # ==================================================
        # CONDITION !=
        # ==================================================

        elif operator == "!=":

            if sample_value != value:

                # feuille finale
                if not isinstance(subtree, dict):

                    return subtree

                # continuer récursivement
                return predict(subtree, sample)

# ==========================================================
# 10. TEST
# ==========================================================

# ----------------------------------------------------------
# Nouvel exemple
# ----------------------------------------------------------

sample = {

    'Meteo': 'Pluie',

    'Temperature': 'Froid',

    'Humidite': 'Normale',

    'Vent': 'Faible'
}

# ----------------------------------------------------------
# Faire prédiction
# ----------------------------------------------------------

prediction = predict(
    tree,
    sample
)

# ==========================================================
# 11. AFFICHAGE RÉSULTAT
# ==========================================================

print("\n===== TEST =====")

print("Exemple :", sample)

print("Classe prédite :", prediction)

===== DATASET =====
     Meteo Temperature Humidite    Vent Jouer
0   Soleil       Chaud    Haute  Faible   Non
1   Soleil       Chaud    Haute    Fort   Non
2  Nuageux       Chaud    Haute  Faible   Oui
3    Pluie       Moyen    Haute  Faible   Oui
4    Pluie       Froid  Normale  Faible   Oui
5    Pluie       Froid  Normale    Fort   Non
6  Nuageux       Froid  Normale    Fort   Oui
7   Soleil       Moyen    Haute  Faible   Non
8   Soleil       Froid  Normale  Faible   Oui
9    Pluie       Moyen  Normale  Faible   Oui

===== SCORES GINI =====
Meteo -> Gini : 0.3167
Temperature -> Gini : 0.4190
Humidite -> Gini : 0.4000
Vent -> Gini : 0.4190

===== SCORES GINI =====
Temperature -> Gini : 0.0000
Humidite -> Gini : 0.0000
Vent -> Gini : 0.3333

===== SCORES GINI =====
Temperature -> Gini : 0.2222
Humidite -> Gini : 0.2500
Vent -> Gini : 0.1667

===== SCORES GINI =====
Temperature -> Gini : inf
Humidite -> Gini : inf

===== ARBRE CART =====
{'Meteo': {'== Soleil': {'Temperature': {'== Fr